In [ ]:
import torch
import sys

In [ ]:
model = torch.hub.load("naver/dune", "dune_vitbase_14_448_paper_encoder")
print(model)

In [ ]:
sys.path.insert(0, "/mnt/homes/@LH-AMIAD/61/dezons_louis-1000009/multimodal-navigation-transformer/dune")
from model.dune import load_dune_encoder_from_checkpoint

ckpt_path = "/mnt/homes/@LH-AMIAD/61/dezons_louis-1000009/multimodal-navigation-transformer/dune/dune_vitbase14_336.pth"
dune_obj = load_dune_encoder_from_checkpoint(ckpt_path)

if isinstance(dune_obj, tuple):
    print(dune_obj)
    dune_obj = dune_obj[0]
    print("dune obj retrieved: ", dune_obj)

im_encoder = dune_obj

num_im_features = 768
im_encoder_type = "dune"

for p in im_encoder.parameters():
    p.requires_grad = False
im_encoder.eval()

In [ ]:
print(im_encoder)
print(type(im_encoder))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
im_encoder = im_encoder.to(device)

device = next(im_encoder.parameters()).device
print(f"Model device: {device}")

In [ ]:
import tarfile
from PIL import Image
import io
from torchvision import transforms

# Function to load image from tar file
def load_image_from_tar(tar_path, image_path_in_tar):
    """
    Extract and load an image from a tar file.
    
    Args:
        tar_path: Path to the tar file
        image_path_in_tar: Path to the image within the tar file
    
    Returns:
        PIL Image object
    """
    with tarfile.open(tar_path, 'r') as tar:
        member = tar.getmember(image_path_in_tar)
        f = tar.extractfile(member)
        img = Image.open(f)
        return img.convert('RGB')

# Example usage
tar_path = "/mnt/homes/@LH-AMIAD/61/robotique/datasets/grand-tour/datas/2024-10-01-11-29-55/images.tar"
image_path_in_tar = "000001.jpg"  # Path within the tar file

# Load image
img = load_image_from_tar(tar_path, image_path_in_tar)
print(f"Image size: {img.size}")

# Preprocess image (adjust based on im_encoder requirements)
preprocess = transforms.Compose([
    transforms.Resize((448, 448)),  # Adjust to your encoder's input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

img_tensor = preprocess(img).unsqueeze(0).to(device)
print(f"Tensor shape: {img_tensor.shape}")

# Feed to encoder
with torch.no_grad():
    encoding = im_encoder(img_tensor)
    print(encoding)
    print(f"Encoding shape: {encoding.shape}")
    print(f"Encoding type: {type(encoding)}")
    print(f"Encoding sample: {encoding[0, :10]}")  # First 10 features